In [ ]:
import os
import numpy as np
import pydicom
from scipy.ndimage import zoom

dataset_path = r"D:\PFA 2eme\manifest-1773666886424\NLST"
save_path = r"D:\PFA 2eme\preprocessed_scans"
os.makedirs(save_path, exist_ok=True)

def get_main_series(year_path):
    """Prendre la série avec le plus de slices (= scan principal)"""
    best_series = None
    max_slices = 0
    for series_folder in os.listdir(year_path):
        series_path = os.path.join(year_path, series_folder)
        if os.path.isdir(series_path):
            dcm_files = [f for f in os.listdir(series_path) if f.endswith('.dcm')]
            if len(dcm_files) > max_slices:
                max_slices = len(dcm_files)
                best_series = series_path
    return best_series

def load_series(series_path):
    """Charger et trier les slices DICOM"""
    dcm_files = sorted([f for f in os.listdir(series_path) if f.endswith('.dcm')])
    slices = [pydicom.dcmread(os.path.join(series_path, f)) for f in dcm_files]
    slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
    volume = np.stack([s.pixel_array for s in slices]).astype(np.float32)
    intercept = slices[0].RescaleIntercept
    slope = slices[0].RescaleSlope
    volume = volume * slope + intercept
    return volume

def normalize_and_resize(volume, target=(64, 128, 128)):
    """Normaliser HU et resize"""
    volume = np.clip(volume, -1000, 400)
    volume = (volume + 1000) / 1400.0
    factors = [t/s for t, s in zip(target, volume.shape)]
    volume = zoom(volume, factors, order=1)
    return volume

def process_patient(patient_id):
    """Traiter un patient → 3 volumes (années 0, 1, 2)"""
    patient_path = os.path.join(dataset_path, str(patient_id))
    year_folders = sorted(os.listdir(patient_path))
    volumes = []
    for year_folder in year_folders:
        year_path = os.path.join(patient_path, year_folder)
        if os.path.isdir(year_path):
            series_path = get_main_series(year_path)
            if series_path:
                volume = load_series(series_path)
                volume = normalize_and_resize(volume)
                volumes.append(volume)
    return np.array(volumes)  # (3, 64, 128, 128)

# Test sur un patient d'abord
print("Test sur patient 100173...")
result = process_patient(100173)
print(f"Shape: {result.shape}")
print(f"Min: {result.min():.3f}, Max: {result.max():.3f}")

In [ ]:
import os
import numpy as np
import pydicom
from scipy.ndimage import zoom
import pandas as pd

# Chemins
dataset_path = r"D:\PFA 2eme\manifest-1773666886424\NLST"
save_path = r"D:\PFA 2eme\preprocessed_scans"
os.makedirs(save_path, exist_ok=True)

# Fonctions preprocessing
def get_main_series(year_path):
    best_series = None
    max_slices = 0
    for series_folder in os.listdir(year_path):
        series_path = os.path.join(year_path, series_folder)
        if os.path.isdir(series_path):
            dcm_files = [f for f in os.listdir(series_path) if f.endswith('.dcm')]
            if len(dcm_files) > max_slices:
                max_slices = len(dcm_files)
                best_series = series_path
    return best_series

def load_series(series_path):
    dcm_files = sorted([f for f in os.listdir(series_path) if f.endswith('.dcm')])
    slices = [pydicom.dcmread(os.path.join(series_path, f)) for f in dcm_files]
    slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
    volume = np.stack([s.pixel_array for s in slices]).astype(np.float32)
    intercept = slices[0].RescaleIntercept
    slope = slices[0].RescaleSlope
    volume = volume * slope + intercept
    return volume

def normalize_and_resize(volume, target=(64, 128, 128)):
    volume = np.clip(volume, -1000, 400)
    volume = (volume + 1000) / 1400.0
    factors = [t/s for t, s in zip(target, volume.shape)]
    volume = zoom(volume, factors, order=1)
    return volume

def process_patient(patient_id):
    patient_path = os.path.join(dataset_path, str(patient_id))
    year_folders = sorted(os.listdir(patient_path))
    volumes = []
    for year_folder in year_folders:
        year_path = os.path.join(patient_path, year_folder)
        if os.path.isdir(year_path):
            series_path = get_main_series(year_path)
            if series_path:
                volume = load_series(series_path)
                volume = normalize_and_resize(volume)
                volumes.append(volume)
    return np.array(volumes)

# Récupérer les PIDs originaux correctement
train_df = pd.read_csv(r"D:\PFA 2eme\preprocessed_clinical\train_clinical.csv")
val_df   = pd.read_csv(r"D:\PFA 2eme\preprocessed_clinical\val_clinical.csv")
test_df  = pd.read_csv(r"D:\PFA 2eme\preprocessed_clinical\test_clinical.csv")

all_pids_csv = pd.concat([train_df, val_df, test_df])['pid'].unique()
nlst_pids = [int(p) for p in os.listdir(dataset_path)
             if os.path.isdir(os.path.join(dataset_path, p))]

# Seulement les PIDs qui ont un vrai dossier NLST
original_pids = [p for p in all_pids_csv if p in nlst_pids]
print(f"Patients originaux à traiter: {len(original_pids)}")

# Preprocessing
success = []
failed = []

for pid in original_pids:
    if os.path.exists(os.path.join(save_path, f"{pid}.npy")):
        success.append(pid)
        print(f" {pid} déjà traité")
        continue
    try:
        result = process_patient(pid)
        if result.shape[0] == 3:
            np.save(os.path.join(save_path, f"{pid}.npy"), result)
            success.append(pid)
            print(f" {pid} → {result.shape}")
        else:
            print(f" {pid} → seulement {result.shape[0]} années")
            failed.append(pid)
    except Exception as e:
        print(f" {pid} → {e}")
        failed.append(pid)

print(f"\n Succès: {len(success)}/{len(original_pids)}")
print(f" Échecs: {len(failed)}")

Patients originaux à traiter: 395
 100173 déjà traité
 100258 déjà traité
 100577 déjà traité
 100718 déjà traité
 100846 déjà traité
 101417 déjà traité
 101428 déjà traité
 101589 déjà traité
 101957 déjà traité
 102110 déjà traité
 102136 déjà traité
 102371 déjà traité
 102410 déjà traité
 102676 déjà traité
 102796 déjà traité
 102987 déjà traité
 103376 déjà traité
 103508 déjà traité
 103545 déjà traité
 103734 déjà traité
 103767 déjà traité
 104023 déjà traité
 104114 déjà traité
 104139 déjà traité
 104207 déjà traité
 104244 déjà traité
 104376 déjà traité
 104519 déjà traité
 104548 déjà traité
 104746 déjà traité
 104779 déjà traité
 104943 déjà traité
 104964 déjà traité
 105081 déjà traité
 105151 déjà traité
 105186 déjà traité
 105307 déjà traité
 105344 déjà traité
 105352 déjà traité
 105437 déjà traité
 105543 déjà traité
 105632 déjà traité
 105713 déjà traité
 105771 déjà traité
 106157 déjà traité
 106229 déjà traité
 106427 déjà traité
 106553 déjà traité
 10659